# GRC-Bench in Colab: pipeline test on the dev split

This notebook checks whether GRC-Bench's own command, `./gb`, runs end to end on a Colab GPU. It is a
**feasibility test, not an evaluation**: dev-split items only, the pinned smoke model (Qwen3-4B Q4_K_M),
placeholder rubrics, and a few items per tier.

| Step | What it checks |
|---|---|
| 1 | the machine: GPU, driver, CUDA version, OS, Python, CPUs, memory, disk |
| 2 | Google Drive, and the GRC-Bench package (`grc-bench-colab.zip`) from your Drive |
| 3 | the sandbox: can **bubblewrap** create one here? |
| 4 | **GNU Radio 3.10.12** from conda-forge (Colab's Ubuntu 22.04 ships 3.10.1) |
| 5 | **GRC-Agent** at the pinned commit `6c98128`, its knowledge databases and the tokenizer. Expected to fail: GRC-Agent is written against Ubuntu 26.04's GNU Radio package, whose GRC layer differs from the upstream 3.10.12.0 that conda-forge builds (found in a rehearsal on the build machine); this step records what Colab says |
| 6 | **llama.cpp b10333** (the pinned build): no prebuilt Linux CUDA download exists, so it is compiled here once and kept in your Drive |
| 7 | `./gb env`: the version pins and the GNU Radio block catalog hash |
| 8 | `./gb e2e`: Tier K (answers scored with placeholder rubrics) and Tier E on a few dev items; Tier F and A too if step 5 worked |
| 9 | speeds: Tier E twice, the GPU's clocks and temperature, Tier-K tokens per second |
| 10 | the judge runner: the same local model grades the Tier-K answers (plumbing only) |
| 11 | results: one folder and one zip in your Drive |

**How to run**
1. Put `grc-bench-colab.zip` in **My Drive/grc-bench-colab/** (no need to unzip it).
2. *Runtime → Change runtime type → **T4 GPU***.
3. *Runtime → **Run all***, and allow Google Drive access when asked.
4. Keep the tab open. **First run: about 1–2 hours**, most of it compiling llama.cpp in step 6. Later runs reuse the build from your Drive.

**What comes back:** `My Drive/grc-bench-colab/results/<run>/results_<run>.zip`, which holds `feasibility.json` (one entry per step: ok, failed or skipped, with times and details), the logs, the e2e report and the run folders.
A step that fails is recorded and the notebook goes on, so one run gives the whole picture.

**What leaves this machine:** nothing is uploaded anywhere except your own Drive. Only dev-split items are in the package; the test split is never in it.

In [ ]:
#@title Settings
DRIVE_FOLDER = "grc-bench-colab"  #@param {type:"string"}
PACKAGE = "grc-bench-colab.zip"  #@param {type:"string"}
TIERS = "K,F,A,E"  #@param {type:"string"}
LIMIT_K = 5  #@param {type:"integer"}
LIMIT_F = 2  #@param {type:"integer"}
LIMIT_A = 1  #@param {type:"integer"}
JUDGE_IN_COLAB = True  #@param {type:"boolean"}
REUSE_LLAMA_BUILD = True  #@param {type:"boolean"}
print(f"tiers {TIERS}: {LIMIT_K} Tier-K questions, {LIMIT_F} Tier-F tasks, {LIMIT_A} Tier-A tasks; "
      f"judge in Colab: {JUDGE_IN_COLAB}; package My Drive/{DRIVE_FOLDER}/{PACKAGE}")

## Helpers
Every command's full output goes to a log file in the results folder; the cell shows the important lines.
`feasibility.json` is written after every step.

In [ ]:
import json, os, re, shutil, subprocess, sys, tarfile, time, zipfile
from pathlib import Path

HOME = Path.home()                                   # /root on Colab: hidden inside GRC-Bench's sandbox
PKG = HOME / "grc-bench-colab"                       # ./gb lives here
GR = Path("/opt/gr")                                 # GNU Radio 3.10.12 (conda-forge)
MM = Path("/opt/micromamba/bin/micromamba")
WORK = HOME / "grc-bench-work"
CACHE = HOME / "grc-bench-cache"
os.environ.update({"GRC_BENCH_CACHE": str(CACHE), "GRC_BENCH_WORK": str(WORK), "GRC_AGENT_REPO": str(HOME / "grc-agent"),
                   "PYTHONDONTWRITEBYTECODE": "1", "QT_QPA_PLATFORM": "offscreen", "MAMBA_ROOT_PREFIX": "/opt/mamba"})
STAMP = time.strftime("%Y%m%d-%H%M%S", time.gmtime())
RESULT = {"notebook": "grcbench_colab_test 1", "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
          "settings": {"tiers": TIERS, "limit_k": LIMIT_K, "limit_f": LIMIT_F, "limit_a": LIMIT_A,
                       "judge_in_colab": JUDGE_IN_COLAB}, "steps": {}}
OUT = Path("/content/grcbench-results") / STAMP       # copied to Drive at each save
OUT.mkdir(parents=True, exist_ok=True)
DRIVE_OUT = None
QUIET = re.compile(r"^\s*$|^(  Downloading|Downloading|Collecting|Building|\[\d+/\d+\] |-- |Resolved |Prepared |Installed |"
                   r"Uninstalled |Audited | \+ | - |Transaction|Linking|Preparing|Verifying|Executing|info|warning libmamba)")


class StopHere(Exception):
    """Stops Run all with a message saying what to do."""


def log(step, msg):
    print(f"[{time.strftime('%H:%M:%S')} | {step}] {msg}", flush=True)


def save():
    RESULT["updated_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    (OUT / "feasibility.json").write_text(json.dumps(RESULT, indent=1))
    if DRIVE_OUT is not None:
        shutil.copytree(OUT, DRIVE_OUT, dirs_exist_ok=True)


def record(step, **kw):
    RESULT["steps"].setdefault(step, {}).update(kw)
    save()


def sh(cmd, step, *, name, cwd=None, env=None, show=60, timeout=None):
    """Run a bash command; full output to OUT/logs/<name>.log; prints at most `show` informative lines and
    the last 15 lines. Returns (exit code, output lines, seconds)."""
    (OUT / "logs").mkdir(exist_ok=True)
    path = OUT / "logs" / f"{name}.log"
    t0, lines, shown = time.time(), [], 0
    with open(path, "w") as fh:
        proc = subprocess.Popen(["bash", "-lc", cmd], cwd=cwd, env={**os.environ, **(env or {})}, text=True,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, errors="replace")
        for line in proc.stdout:
            fh.write(line)
            lines.append(line.rstrip("\n"))
            if shown < show and not QUIET.search(line):
                print("    " + line.rstrip()[:220], flush=True)
                shown += 1
        try:
            code = proc.wait(timeout=timeout)
        except subprocess.TimeoutExpired:
            proc.kill()
            code = -9
    if shown >= show:
        print("    ... (full output in " + str(path) + "); last lines:")
        for line in lines[-15:]:
            print("    " + line[:220])
    secs = round(time.time() - t0, 1)
    log(step, f"exit {code} after {secs} s: {cmd[:100]}")
    return code, lines, secs


def step(key, title, fn):
    """Run one step; record ok / failed (with the error) and its time. Later steps still run."""
    print("=" * 100 + f"\n{key}: {title}\n" + "=" * 100, flush=True)
    t0 = time.time()
    try:
        detail = fn() or {}
        record(key, title=title, status="ok", seconds=round(time.time() - t0, 1), **detail)
    except StopHere:
        record(key, title=title, status="failed", seconds=round(time.time() - t0, 1))
        raise
    except Exception as exc:
        record(key, title=title, status="failed", seconds=round(time.time() - t0, 1), error=f"{type(exc).__name__}: {exc}")
        log(key, f"FAILED: {type(exc).__name__}: {exc}  (recorded; the notebook goes on)")


def gb(args, step_key, *, name, show=60, timeout=None):
    return sh(f"./gb {args}", step_key, name=name, cwd=PKG, show=show, timeout=timeout)

print("helpers ready; results go to", OUT)

## Step 1: the machine

In [ ]:
def machine():
    if "google.colab" not in sys.modules:
        raise StopHere("This notebook is written for Google Colab.")
    if not shutil.which("nvidia-smi"):
        raise StopHere("No GPU: Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all.")
    q = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total,compute_cap", "--format=csv,noheader"],
                       capture_output=True, text=True).stdout.strip().split(", ")
    head = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
    cuda = re.search(r"CUDA Version:\s*([\d.]+)", head)
    osr = dict(l.rstrip().split("=", 1) for l in open("/etc/os-release") if "=" in l)
    info = {"gpu": q[0], "driver": q[1], "gpu_memory": q[2], "compute_cap": q[3],
            "driver_cuda": cuda.group(1) if cuda else None, "os": osr["PRETTY_NAME"].strip('"'),
            "python": sys.version.split()[0], "cpus": os.cpu_count(),
            "memory_gb": round(int(re.search(r"MemTotal:\s+(\d+)", open("/proc/meminfo").read()).group(1)) / 1024 ** 2, 1),
            "free_disk_gb": round(shutil.disk_usage("/content").free / 1e9), "user": os.environ.get("USER") or os.getuid(),
            "kernel": os.uname().release}
    for k, v in info.items():
        log("step 1", f"{k:13s} {v}")
    return info

step("1_machine", "the machine", machine)
GPU = RESULT["steps"]["1_machine"]

## Step 2: Google Drive and the GRC-Bench package

In [ ]:
def package():
    global DRIVE_OUT
    from google.colab import drive
    drive.mount("/content/drive")
    root = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    zpath = root / PACKAGE
    if not zpath.exists():
        raise StopHere(f"{zpath} not found: put {PACKAGE} in My Drive/{DRIVE_FOLDER}/ and Run all again.")
    gpu = re.sub(r"[^A-Za-z0-9]+", "-", GPU["gpu"]).strip("-")
    DRIVE_OUT = root / "results" / f"{STAMP}_{gpu}"
    if PKG.exists():
        shutil.rmtree(PKG)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(HOME)
    for p in [PKG / "gb", *PKG.rglob("*.sh")]:
        p.chmod(0o755)
    version = (PKG / "VERSION.txt").read_text().splitlines()[:3]
    for line in version:
        log("step 2", line)
    return {"package": str(zpath), "package_bytes": zpath.stat().st_size, "version": version, "drive_results": str(DRIVE_OUT)}

step("2_package", "Google Drive and the package", package)

## Step 3: the sandbox (bubblewrap)
GRC-Bench runs every model-written flowgraph and agent command inside bubblewrap. Colab runs as root inside
a container, so bubblewrap may only work with a user namespace of its own (`--unshare-user`); `./gb` then uses
that (`GRC_BENCH_BWRAP_USERNS=1`).

In [ ]:
def sandbox():
    code, _, _ = sh("apt-get -qq update >/dev/null && apt-get -qq install -y bubblewrap >/dev/null && bwrap --version",
                    "step 3", name="03_bubblewrap_install")
    base = "bwrap --ro-bind / / --dev /dev --proc /proc --tmpfs /tmp --unshare-pid --unshare-net --die-with-parent --new-session"
    probes = {}
    for label, extra in (("plain", ""), ("with --unshare-user", " --unshare-user")):
        c, out, _ = sh(f"{base}{extra} -- bash -c 'echo inside; cat /proc/self/uid_map'", "step 3", name=f"03_probe_{label.split()[-1]}")
        probes[label] = {"exit": c, "output": out[-3:]}
    if probes["plain"]["exit"] == 0:
        mode = "plain"
    elif probes["with --unshare-user"]["exit"] == 0:
        mode = "user namespace"
        os.environ["GRC_BENCH_BWRAP_USERNS"] = "1"
    else:
        mode = "none"
    log("step 3", f"sandbox: {mode}")
    if mode == "none":
        raise RuntimeError("bubblewrap cannot create a sandbox here: Tier F (V2 and up) and Tier A cannot run")
    return {"install_exit": code, "probes": probes, "mode": mode}

step("3_sandbox", "bubblewrap", sandbox)

## Step 4: GNU Radio 3.10.12 (conda-forge)

In [ ]:
def gnuradio():
    if not MM.exists():
        sh("mkdir -p /opt/micromamba && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /opt/micromamba bin/micromamba",
           "step 4", name="04_micromamba")
    code, _, secs = sh(f"{MM} create -y -q -p {GR} -c conda-forge python=3.14 gnuradio=3.10.12.0", "step 4",
                       name="04_gnuradio_create", show=20)
    if code:
        raise RuntimeError("micromamba could not install gnuradio=3.10.12.0 (log 04_gnuradio_create.log)")
    out = subprocess.run([str(GR / "bin/python"), "-c", "from gnuradio import gr; print(gr.version())"], capture_output=True, text=True)
    version = out.stdout.strip() or out.stderr.strip()[-300:]
    # GRC-Bench reads the block catalog and examples from /usr/share/gnuradio and runs grcc from PATH
    links = {}
    share = Path("/usr/share/gnuradio")
    if share.exists() and not share.is_symlink():
        share.rename("/usr/share/gnuradio.ubuntu")
    if not share.exists():
        share.symlink_to(GR / "share/gnuradio")
    links["/usr/share/gnuradio"] = str(share.resolve())
    for tool in ("grcc", "gnuradio-companion", "gnuradio-config-info"):
        target, link = GR / "bin" / tool, Path("/usr/local/bin") / tool
        if target.exists():
            link.unlink(missing_ok=True)
            link.symlink_to(target)
            links[str(link)] = str(target)
    log("step 4", f"GNU Radio {version} in {secs} s")
    return {"gnuradio_version": version, "install_seconds": secs, "links": links}

step("4_gnuradio", "GNU Radio 3.10.12 (conda-forge)", gnuradio)

## Step 5: GRC-Agent at `6c98128`

In [ ]:
def grc_agent():
    repo = HOME / "grc-agent"
    sh("pip install -q uv==0.12.21 && uv --version", "step 5", name="05_uv")
    if not (repo / ".git").exists():
        sh(f"git clone -q https://github.com/qoherent/grc-agent.git {repo}", "step 5", name="05_clone")
    c, out, _ = sh(f"git -C {repo} checkout -q 6c98128 && git -C {repo} rev-parse HEAD", "step 5", name="05_checkout")
    head = out[-1] if out else ""
    c, _, secs = sh(f"cd {repo} && uv venv -q --system-site-packages --python {GR}/bin/python .venv && "
                    "uv sync -q --extra dev --locked --python .venv/bin/python", "step 5", name="05_uv_sync", show=30)
    if c:
        raise RuntimeError("uv sync failed (log 05_uv_sync.log)")
    py = repo / ".venv/bin/python"
    imports = subprocess.run([str(py), "-c", "import grc_agent, gnuradio, pydantic_ai; from gnuradio import gr; print(gr.version())"],
                             capture_output=True, text=True)
    c_db, db_out, _ = sh(f"cd {repo} && GRC_EMBED_BACKEND=lexical {py} -c \"from grc_agent.adapter import rag\n"
                    "for d in ('catalog', 'docs'):\n    p, m = rag.get_db_and_model(d)\n    rag._ensure_db_built(d, p, m)\"",
                    "step 5", name="05_knowledge_dbs")
    (CACHE / "tiktoken").mkdir(parents=True, exist_ok=True)
    c_tok, _, _ = sh(f"TIKTOKEN_CACHE_DIR={CACHE}/tiktoken {py} -c \"import tiktoken; tiktoken.get_encoding('cl100k_base')\"",
                     "step 5", name="05_tiktoken")
    vec = repo / "src/grc_agent/vectors"
    return {"head": head, "sync_seconds": secs, "imports": (imports.stdout or imports.stderr).strip()[-300:],
            "knowledge_dbs": c_db == 0 and (vec / "catalog_lexical.db").exists() and (vec / "docs_lexical.db").exists(),
            "knowledge_dbs_error": None if c_db == 0 else db_out[-1:], "tokenizer": c_tok == 0}

step("5_grc_agent", "GRC-Agent 6c98128", grc_agent)

## Step 6: llama.cpp b10333 with CUDA
`./gb fetch` downloads the pinned CPU build, the smoke model and the embedding model (each checked by SHA-256).
llama.cpp publishes no Linux CUDA build of b10333, so `./gb fetch --cuda` compiles it for this GPU
(the first time: up to an hour or more on Colab's 2 CPUs) and the result is kept in your Drive for the next run.
A build placed in `My Drive/grc-bench-colab/cache/` beforehand (`llama-b10333-cuda-sm75-cu12.4.tar.gz` for a T4) is used instead.

In [ ]:
def llama():
    c, _, fetch_s = gb("fetch", "step 6", name="06_fetch")
    if c:
        raise RuntimeError("./gb fetch failed (log 06_fetch.log)")
    cc = GPU["compute_cap"].replace(".", "")
    drv = tuple(int(x) for x in (GPU["driver_cuda"] or "12.4").split("."))
    cuda = "12.9" if drv >= (12, 9) else f"{drv[0]}.{drv[1]}"
    out_dir = CACHE / "llama.cpp" / "b10333-cuda"
    cache = Path("/content/drive/MyDrive") / DRIVE_FOLDER / "cache"
    saved = cache / f"llama-b10333-cuda-sm{cc}-cu{cuda}.tar.gz"
    # a build kept earlier (or put there by hand) for this GPU, made with a CUDA version this driver supports
    cu_of = lambda p: tuple(int(x) for x in re.search(r"-cu(\d+)\.(\d+)\.tar\.gz$", p.name).groups())
    usable = [p for p in cache.glob(f"llama-b10333-cuda-sm{cc}-cu*.tar.gz") if cu_of(p) <= drv[:2]]
    if usable:
        saved = max(usable, key=cu_of)
    how, build_s = "built", None
    if REUSE_LLAMA_BUILD and saved.exists():
        out_dir.parent.mkdir(parents=True, exist_ok=True)
        with tarfile.open(saved) as t:
            t.extractall(out_dir.parent)
        how = f"reused from Drive ({saved.name})"
    else:
        gcc = "13" if drv >= (12, 6) else "12"           # conda-forge's nvcc below 12.6 accepts GCC 12 at most
        c, _, build_s = sh(f"CUDA_ARCH={cc} CUDA_VERSION={cuda} GCC_VERSION={gcc} SYSROOT=2.28 JOBS={os.cpu_count() or 2} ./gb fetch --cuda",
                           "step 6", name="06_build_cuda", cwd=PKG, show=40)
        if c:
            raise RuntimeError("the CUDA build failed (log 06_build_cuda.log)")
        saved.parent.mkdir(parents=True, exist_ok=True)
        with tarfile.open(saved, "w:gz") as t:
            t.add(out_dir, arcname=out_dir.name)
    env = {"LD_LIBRARY_PATH": f"{out_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}"}
    v = subprocess.run([str(out_dir / "llama-server"), "--version"], capture_output=True, text=True, env={**os.environ, **env})
    version = (v.stdout + v.stderr).strip().splitlines()[-2:]
    log("step 6", f"llama.cpp {how}: {version}")
    return {"fetch_seconds": fetch_s, "cuda_arch": cc, "cuda_version": cuda, "how": how, "build_seconds": build_s,
            "version": version, "build_cache": str(saved)}

step("6_llama_cpp", "llama.cpp b10333 (CUDA)", llama)

## Step 7: `./gb env`, the pins and the block catalog

In [ ]:
def pins():
    c, out, _ = gb("env", "step 7", name="07_env")
    env = json.loads((PKG / "reports/environment.json").read_text())
    py = HOME / "grc-agent/.venv/bin/python"
    cat = subprocess.run([str(py), "-c", "from grcbench.tier_f.ladder import pinned_catalog_hash, catalog_matches_pin; "
                          "print(pinned_catalog_hash(), catalog_matches_pin())"], cwd=PKG, capture_output=True, text=True,
                         env={**os.environ, "PYTHONPATH": str(PKG / "src")})
    parts = cat.stdout.split()
    shutil.copy(PKG / "reports/environment.json", OUT / "environment.json")
    return {"pins_ok": c == 0 and any("pins ok" in l for l in out), "pin_message": out[-1] if out else "",
            "gnuradio_version": env.get("gnuradio_version"), "grc_agent_commit": env.get("grc_agent_commit"),
            "python": env.get("python"), "packages": env.get("packages"),
            "catalog_hash": parts[0] if parts else cat.stderr[-300:], "catalog_matches_pin": parts[1:] == ["True"]}

step("7_pins", "./gb env, pins and catalog", pins)

## Step 8: `./gb e2e` on a few dev items
One llama-server (the smoke model, every layer on the GPU) serves Tier K, F and A in turn; then Tier E runs
llama-bench. Tier-K answers are scored with the **placeholder rubrics** (the training release's reference answer as
the only point): that tests the routing, not the model.

In [ ]:
E_NAME = "colab_" + re.sub(r"[^a-z0-9]+", "_", GPU["gpu"].lower()).strip("_")
E2E = WORK / "colab_e2e"

def e2e():
    tiers = [t.strip().upper() for t in TIERS.split(",") if t.strip()]
    agent_ok = RESULT["steps"].get("5_grc_agent", {}).get("knowledge_dbs") is True
    sandbox_ok = RESULT["steps"].get("3_sandbox", {}).get("status") == "ok"
    skipped = [t for t in tiers if t in "FA" and not (agent_ok and sandbox_ok)]
    if skipped:
        log("step 8", f"Tier {', '.join(skipped)} not run: GRC-Agent (step 5) or the sandbox (step 3) did not work here")
    tiers = [t for t in tiers if t not in skipped]
    args = (f"e2e --out {E2E} --only {','.join(tiers)} --limit-k {LIMIT_K} --limit-f {LIMIT_F} --limit-a {LIMIT_A} "
            f"--rubrics colab/dev_placeholder.jsonl --report {OUT}/e2e_colab.md --e-name {E_NAME} "
            f"--machine-label 'Google Colab ({GPU['gpu']}, driver {GPU['driver']})'")
    c, _, secs = gb(args, "step 8", name="08_e2e", show=80)
    s = json.loads((E2E / "summary.json").read_text())
    for k, v in s.get("tiers", {}).items():
        log("step 8", f"Tier {k}: " + json.dumps({x: v.get(x) for x in ("items", "answers", "n_rules", "n_judge", "tasks",
                                                                     "samples", "episodes", "passed", "funnel",
                                                                     "generation_tps", "error", "seconds") if x in v}))
    return {"exit": c, "seconds": secs, "tiers_run": tiers, "tiers_not_run": skipped, "tiers": s.get("tiers"),
            "server": s.get("server")}

step("8_e2e", "./gb e2e on the dev split", e2e)

## Step 9: can speeds be recorded accurately here?
Tier E runs a second time with the same settings. Two runs that agree, a small spread inside each run, and steady
GPU clocks mean the recorded speed is a property of the setup, not of the moment. Colab GPUs are shared and can be
throttled, so this is worth checking.

In [ ]:
def smi():
    q = "clocks.sm,clocks.max.sm,clocks.mem,temperature.gpu,power.draw,pstate,clocks_throttle_reasons.active"
    return subprocess.run(["nvidia-smi", f"--query-gpu={q}", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()

def speeds():
    model, llama_dir = CACHE / "models/Qwen3-4B-Q4_K_M.gguf", CACHE / "llama.cpp/b10333-cuda"
    before = smi()
    c, _, secs = gb(f"tier-e run --model {model} --hardware H3 --llama-dir {llama_dir} --contexts 4096 --purpose plumbing "
                    f"--name {E_NAME}_repeat --machine-label 'Google Colab ({GPU['gpu']})' "
                    "--note 'Colab pipeline test: repeat of the e2e Tier-E run, to check that speeds repeat'",
                    "step 9", name="09_tier_e_repeat")
    after = smi()
    runs = {}
    for name in (E_NAME, E_NAME + "_repeat"):
        p = PKG / "reports/tier_e" / f"{name}.json"
        if p.exists():
            rec = json.loads(p.read_text())
            shutil.copy(p, OUT / p.name)
            ctx = next((x for x in rec.get("contexts", []) if x.get("context") == 4096), {})
            runs[name] = {k: {f: (ctx.get(k) or {}).get(f) for f in ("median_ts", "min_ts", "max_ts", "samples_ts",
                                                                     "llama_bench_stddev_ts")}
                          for k in ("generation", "prompt") if ctx.get(k)}
    gen = [r["generation"]["median_ts"] for r in runs.values() if r.get("generation")]
    k_rows = [json.loads(l) for l in open(E2E / "tier_k/answers.jsonl")] if (E2E / "tier_k/answers.jsonl").exists() else []
    k_tps = [round(r["usage"]["completion_tokens"] / r["seconds"], 1) for r in k_rows
             if r.get("usage") and r.get("seconds") and r["usage"].get("completion_tokens")]
    out = {"exit": c, "runs": runs, "smi_before": before, "smi_after": after, "tier_k_answer_tps": k_tps}
    if len(gen) == 2:
        out["generation_tps_two_runs"] = gen
        out["difference_pct"] = round(100 * abs(gen[0] - gen[1]) / max(gen), 2)
    log("step 9", f"generation tokens/s: {gen}; Tier-K answers: {k_tps}")
    log("step 9", f"GPU before: {before}")
    log("step 9", f"GPU after:  {after}")
    return out

step("9_speeds", "speeds: Tier E twice, GPU clocks", speeds)

## Step 10: the judge runner (plumbing)
The real judge is a hosted model (not available yet). Here the smoke model grades its own Tier-K answers through
`./gb tier-k judge`, only to show that the requests, verdicts and scores flow. These scores mean nothing.

In [ ]:
JUDGE = r"""
import json, sys
from pathlib import Path
from grcbench.tier_a import smoke
from grcbench.tier_k import judge_run
out = Path(sys.argv[1])
ctx = smoke.server_context(smoke.native_context(smoke.SMOKE_MODEL))
with smoke.LocalServer(out / "judge-server.log", ctx=ctx) as url:
    ep = judge_run.Endpoint(base_url=url, model=smoke.SERVER_ALIAS)
    s = judge_run.judge(out, primary=ep, fallback=ep, timeout=1800)
print(json.dumps({k: s.get(k) for k in ("judge", "n_judged", "n_judge_failed", "n_judge_request_failed", "report_counts")}))
"""

def judge():
    if not JUDGE_IN_COLAB:
        return {"skipped": "JUDGE_IN_COLAB is off"}
    (OUT / "judge_plumbing.py").write_text(JUDGE)
    c, out, secs = sh(f"PYTHONPATH={PKG}/src {HOME}/grc-agent/.venv/bin/python {OUT}/judge_plumbing.py {E2E}/tier_k",
                      "step 10", name="10_judge", cwd=PKG)
    last = next((l for l in reversed(out) if l.startswith("{")), None)
    return {"exit": c, "seconds": secs, "summary": json.loads(last) if last else None}

step("10_judge", "the judge runner (plumbing)", judge)

## Step 11: results

In [ ]:
def results():
    for src, dst in ((E2E, "e2e_run"), (PKG / "runs", "runs")):
        if Path(src).exists():
            shutil.copytree(src, OUT / dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns("*.gguf", "home"))
    RESULT["finished_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    save()
    z = OUT.parent / f"results_{OUT.name}.zip"
    shutil.make_archive(str(z.with_suffix("")), "zip", OUT)
    if DRIVE_OUT is not None:
        shutil.copy(z, DRIVE_OUT / z.name)
    return {"zip": str((DRIVE_OUT or OUT) / z.name)}

step("11_results", "results", results)
print("\n" + "=" * 100)
for k, v in RESULT["steps"].items():
    print(f"{k:14s} {v.get('status', '?'):7s} {v.get('seconds', '')!s:>8} s  {v.get('error', '')[:120]}")
print("=" * 100)
print("Download this file and put it in your Downloads folder:", RESULT["steps"].get("11_results", {}).get("zip"))